In [1]:
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim.lr_scheduler as lr_scheduler

# ==========================================
# 1. DATASET GENERATION (Cybersecurity Logic)
# ==========================================
def calculate_cost(sequence):
    """
    Simulates the database API with vulnerabilities.
    Calculates the accumulated cost in milliseconds with 1 and 2-step memory.
    """
    if not sequence: return 0.0
    
    total_cost = 0.0
    h1 = None  # t-1
    h2 = None  # t-2
    
    for char in sequence:
        cost = 5.0  # Generic base cost
        
        # --- 2-STEP RULES ---
        if h2 == 'I' and h1 == 'W' and char == 'B':
            cost = 40.0  # Dump block
        elif h2 == 'B' and h1 == 'R' and char == 'I':
            cost = 70.0  # Cache miss
            
        # --- 1-STEP RULES ---
        elif h1 == 'W' and char == 'I':
            cost = 45.0  # Local Maximum Decoy
        elif h1 == 'W' and char == 'B':
            cost = 15.0  # Noise
        elif h1 == 'R' and char == 'I':
            cost = 22.0  # Noise
        elif h1 == 'R' and char == 'W':
            cost = 15.0  # Noise
        elif h1 == 'B' and char == 'I':
            cost = 20.0  # Noise
        elif h1 == 'I' and char == 'R':
            cost = 12.0  # Noise
            
        total_cost += cost
        h2 = h1
        h1 = char
        
    return float(total_cost)

# Generate 20,000 random words (More data to catch rare rules)
data = []
for _ in range(20000):
    length = random.randint(1, 12)
    sequence = "".join(random.choices(['W', 'I', 'B', 'R'], k=length))
    data.append({'sequence': sequence, 'cost': calculate_cost(sequence)})

df = pd.DataFrame(data)

# ==========================================
# 2. DATA PREPARATION
# ==========================================
vocab = {'W': 1, 'I': 2, 'B': 3, 'R': 4}

class SequenceDataset(Dataset):
    def __init__(self, dataframe):
        self.X = [torch.tensor([vocab[char] for char in seq], dtype=torch.long) for seq in dataframe['sequence']]
        self.Y = torch.tensor(dataframe['cost'].values, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def collate_fn(batch):
    X, Y = zip(*batch)
    lengths = torch.tensor([len(x) for x in X], dtype=torch.long)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.stack(Y), lengths

dataset = SequenceDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

# ==========================================
# 3. MODEL 
# ==========================================
class CostPredictorRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CostPredictorRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # A simple network with ReLU is enough to emulate conditional summation
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

# Vocabulary of 5 (4 letters + padding)
model = CostPredictorRNN(vocab_size=5, embed_dim=8, hidden_dim=32)

criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ==========================================
# 4. TRAINING
# ==========================================
epochs = 200 
print("Starting RNN training (API Audit)...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for X_batch, Y_batch, lengths in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch, lengths)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:03d}/{epochs} - Loss (MAE): {avg_loss:.4f} | LR: {current_lr:.6f}")

# ==========================================
# 5. ADVERSARIAL TEST
# ==========================================
print("\nTesting Predictions (Adversarial Attacks):")
model.eval()

# Evaluate different types of attacks
examples = [
    "WIWIWIWI",      # Fuzzer: The local maximum (8 letters)
    "IWBRIWBR",      # Cascade: The true global maximum (8 letters)
    "BBBB",          # Harmless flow
    "WRBIBRIW"       # Complex random noise
]

with torch.no_grad():
    for seq in examples:
        tensor_in = torch.tensor([vocab[c] for c in seq], dtype=torch.long).unsqueeze(0)
        length = torch.tensor([len(seq)], dtype=torch.long)
        pred = model(tensor_in, length).item()
        actual_cost = calculate_cost(seq)
        print(f"Sequence: {seq:<10} | RNN predicts: {pred:6.1f} | Actual cost: {actual_cost:6.1f}")

C:\Users\rafam\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting RNN training (API Audit)...
Epoch 001/200 - Loss (MAE): 20.2432 | LR: 0.010000
Epoch 010/200 - Loss (MAE): 3.4516 | LR: 0.010000
Epoch 020/200 - Loss (MAE): 2.6698 | LR: 0.010000
Epoch 030/200 - Loss (MAE): 2.2518 | LR: 0.010000
Epoch 040/200 - Loss (MAE): 1.9026 | LR: 0.010000
Epoch 050/200 - Loss (MAE): 0.9032 | LR: 0.005000
Epoch 060/200 - Loss (MAE): 0.9199 | LR: 0.005000
Epoch 070/200 - Loss (MAE): 0.4531 | LR: 0.002500
Epoch 080/200 - Loss (MAE): 0.2360 | LR: 0.001250
Epoch 090/200 - Loss (MAE): 0.2115 | LR: 0.001250
Epoch 100/200 - Loss (MAE): 0.1112 | LR: 0.000625
Epoch 110/200 - Loss (MAE): 0.0458 | LR: 0.000156
Epoch 120/200 - Loss (MAE): 0.0401 | LR: 0.000156
Epoch 130/200 - Loss (MAE): 0.0261 | LR: 0.000078
Epoch 140/200 - Loss (MAE): 0.0212 | LR: 0.000039
Epoch 150/200 - Loss (MAE): 0.0204 | LR: 0.000039
Epoch 160/200 - Loss (MAE): 0.0178 | LR: 0.000020
Epoch 170/200 - Loss (MAE): 0.0174 | LR: 0.000020
Epoch 180/200 - Loss (MAE): 0.0176 | LR: 0.000020
Epoch 190/20

In [9]:
import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=0.5, rounding_step=1)

params = QuantitativeObservationTableParameters(
    tol_dist_init=0.7, 
    rec_method="direct", 
    r=3, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent_det()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_deterministic_wfa_opt()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=6, 
        random_max_len=15, 
        num_random=1000,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1


--- Iteration 1 ---
[Exact L* Synthesis] Empirical Hankel DWFA successfully built with 1 states.
[OK] The Automaton satisfies the Twins Property. It is 100% determinizable!
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 6...
      -> Failure detail for 'WI': RNN=50.00, WFA=10.00

[!] Counterexample found (Exhaustive): 'WI'
[+] Processing Counterexample: 'WI' | S size: 1
[Counterexample] Added ALL suffixes to E from: 'WI'

--- Iteration 2 ---
[Exact L* Synthesis] Empirical Hankel DWFA successfully built with 5 states.
[OK] The Automaton satisfies the Twins Property. It is 100% determinizable!
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 6...
      -> Failure detail for 'IR': RNN=17.00, WFA=10.00

[!] Counterexample found (Exhaustive): 'IR'
[+] Processing Counterexample: 'IR' | S size: 5
[Counterexample] Added ALL suffixes to E from: 'IR'

--- Iteration 3 ---
[Exact L* Synthesis] Empirical Hankel DWFA successfully built with 6 states.
[OK] The Automaton satisfies the Twi

In [13]:
from graphviz import Digraph
import numpy as np

def draw_tropical_wfa(wfa: 'TropicalWFA', name="TropicalWFA"):
    """
    Generates a Graphviz diagram for a TropicalWFA object,
    formatting weights to a maximum of 3 decimal places.
    """
    dot = Digraph(name)
    dot.attr(rankdir='LR') # Left-to-right orientation
    
    n_states = len(wfa.q0)
    
    # 1. Create nodes (states) and initial/final transitions
    for i in range(n_states):
        label = f"S{i}"
        
        # Process final weights (beta)
        is_final = wfa.final[i] != float('-inf')
        if is_final:
            # Se aplica round() para limitar a 3 decimales
            label += f"\n[β={round(wfa.final[i], 3)}]"
            # Double circle for final states
            dot.node(str(i), label, shape="doublecircle")
        else:
            # Normal circle for the rest
            dot.node(str(i), label, shape="circle")
        
        # Process initial weights (alpha) - Incoming arrows
        if wfa.q0[i] != float('-inf'):
            # Invisible ghost node for the incoming arrow
            dot.node(f"start{i}", "", shape="none", width="0", height="0")
            # Se aplica round() al peso inicial
            dot.edge(f"start{i}", str(i), label=f"α={round(wfa.q0[i], 3)}")

    # 2. Create internal transitions (delta)
    for char in wfa.alphabet:
        matrix = wfa.delta[char]
        for i in range(n_states):
            for j in range(n_states):
                weight = matrix[i, j]
                # If the weight is not -inf, there is a transition
                if weight != float('-inf'):
                    # Se aplica round() al peso de la transición
                    dot.edge(str(i), str(j), label=f"{char}:{round(weight, 3)}")
                    
    return dot


In [31]:
dot = draw_tropical_wfa(wfa)
dot.render('NoiseCase3', view=True,format='png')

'NoiseCase3.png'

In [29]:
#Manipulate weights

wfa.push_weights_to_positive()
wfa.normalize_global_bias()
wfa.push_final_weights_to_zero()


[+] Weight Pushing applied: Negative weights pushed towards the endpoints.
[+] Bias Normalized: Global weight shifted by -50.42.
[+] Backward Weight Pushing: Final weights absorbed to 0.0.


In [21]:
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim.lr_scheduler as lr_scheduler

# ==========================================
# 1. DATASET GENERATION (Cybersecurity with Network Latency)
# ==========================================
def calculate_sequence_costs(sequence):
    """
    Simulates the API with vulnerabilities + Stochastic Network/CPU Latency.
    """
    if not sequence: return 0.0, 0.0
    
    total_clean_cost = 0.0
    total_noisy_cost = 0.0
    h1 = None  # t-1
    h2 = None  # t-2
    
    for char in sequence:
        mu = 5.0  # Generic base cost
        
        # --- 2-STEP RULES ---
        if h2 == 'I' and h1 == 'W' and char == 'B': mu = 40.0
        elif h2 == 'B' and h1 == 'R' and char == 'I': mu = 70.0
            
        # --- 1-STEP RULES ---
        elif h1 == 'W' and char == 'I': mu = 45.0
        elif h1 == 'W' and char == 'B': mu = 15.0
        elif h1 == 'R' and char == 'I': mu = 22.0
        elif h1 == 'R' and char == 'W': mu = 15.0
        elif h1 == 'B' and char == 'I': mu = 20.0
        elif h1 == 'I' and char == 'R': mu = 12.0
        
        # Inject noise: Heavier queries fluctuate more
        sigma = 0.5 + (0.10 * mu)
        operation_time = max(0.0, random.gauss(mu, sigma))
        
        total_clean_cost += mu
        total_noisy_cost += operation_time
        
        h2 = h1
        h1 = char
        
    return float(total_noisy_cost), float(total_clean_cost)

data = []
for _ in range(20000): # Keeping 20k because 2-step rules are rarer
    length = random.randint(1, 12)
    sequence = "".join(random.choices(['W', 'I', 'B', 'R'], k=length))
    c_noisy, c_clean = calculate_sequence_costs(sequence)
    
    data.append({
        'sequence': sequence, 
        'noisy_cost': c_noisy,       # Target for the RNN
        'clean_real_cost': c_clean   # Ground truth
    })

df = pd.DataFrame(data)

print("Dataset Sample (Audit with Real Latency):")
try:
    display(df.head())
except NameError:
    print(df.head())

# ==========================================
# 2. DATA PREPARATION
# ==========================================
vocab = {'W': 1, 'I': 2, 'B': 3, 'R': 4}

class CyberSequenceDataset(Dataset):
    def __init__(self, dataframe):
        self.X = [torch.tensor([vocab[char] for char in seq], dtype=torch.long) for seq in dataframe['sequence']]
        self.Y = torch.tensor(dataframe['noisy_cost'].values, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def collate_fn(batch):
    X, Y = zip(*batch)
    lengths = torch.tensor([len(x) for x in X], dtype=torch.long)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.stack(Y), lengths

dataset = CyberSequenceDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# ==========================================
# 3. MODEL 
# ==========================================
class CostPredictorRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CostPredictorRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Increased to 64 neurons. Memorizing 2nd order rules with noise requires more capacity.
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

model = CostPredictorRNN(vocab_size=5, embed_dim=8, hidden_dim=64)
criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ==========================================
# 4. TRAINING (NOISE FILTERING)
# ==========================================
epochs = 150 
print("\nStarting training (Distilling vulnerabilities under noise)...")

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    for X_batch, Y_batch, lengths in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch, lengths)
        if predictions.dim() == 0: predictions = predictions.unsqueeze(0)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_test_loss = 0
    with torch.no_grad():
        for X_batch_test, Y_batch_test, lengths_test in test_loader:
            test_predictions = model(X_batch_test, lengths_test)
            if test_predictions.dim() == 0: test_predictions = test_predictions.unsqueeze(0)
            test_loss = criterion(test_predictions, Y_batch_test)
            total_test_loss += test_loss.item()
            
    avg_test_loss = total_test_loss / len(test_loader)
    scheduler.step(avg_test_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:03d}/{epochs} - Train MAE: {avg_train_loss:.4f} | Test MAE: {avg_test_loss:.4f} | LR: {current_lr:.6f}")

# ==========================================
# 5. ADVERSARIAL TEST 
# ==========================================
print("\nTesting Predictions (Network vs Real Vulnerabilities):")
model.eval()

# Evaluate key attacks
examples = [
    "WIWIWIWI",      
    "IWBRIWBR",      
    "BBBB",          
    "WRBIBRIW"       
]

with torch.no_grad():
    for seq in examples:
        tensor_in = torch.tensor([vocab[c] for c in seq], dtype=torch.long).unsqueeze(0)
        length = torch.tensor([len(seq)], dtype=torch.long)
        
        pred = model(tensor_in, length).item()
        
        # Recover clean truth to verify if the network filtered the noise
        _, clean_real = calculate_sequence_costs(seq)
        
        print(f"Sequence: {seq:<10} | Clean Truth: {clean_real:6.1f} | RNN Prediction: {pred:6.1f}")

Dataset Sample (Audit with Real Latency):


,sequence,noisy_cost,clean_real_cost
0,WBRBRWWBR,79.707884,75.0
1,WBWIR,76.640681,82.0
2,RRRBB,26.858154,25.0
3,IRRRW,39.650869,42.0
4,RWBRRBR,56.416803,55.0


C:\Users\rafam\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Starting training (Distilling vulnerabilities under noise)...
Epoch 001/150 - Train MAE: 18.1993 | Test MAE: 10.0970 | LR: 0.010000
Epoch 010/150 - Train MAE: 5.7209 | Test MAE: 4.9517 | LR: 0.010000
Epoch 020/150 - Train MAE: 4.1880 | Test MAE: 4.1440 | LR: 0.005000
Epoch 030/150 - Train MAE: 3.8431 | Test MAE: 4.0204 | LR: 0.002500
Epoch 040/150 - Train MAE: 3.7056 | Test MAE: 3.6714 | LR: 0.001250
Epoch 050/150 - Train MAE: 3.6191 | Test MAE: 3.6502 | LR: 0.000625
Epoch 060/150 - Train MAE: 3.5828 | Test MAE: 3.6727 | LR: 0.000313
Epoch 070/150 - Train MAE: 3.5366 | Test MAE: 3.6343 | LR: 0.000078
Epoch 080/150 - Train MAE: 3.5288 | Test MAE: 3.6416 | LR: 0.000020
Epoch 090/150 - Train MAE: 3.5251 | Test MAE: 3.6511 | LR: 0.000010
Epoch 100/150 - Train MAE: 3.5243 | Test MAE: 3.6428 | LR: 0.000002
Epoch 110/150 - Train MAE: 3.5238 | Test MAE: 3.6433 | LR: 0.000001
Epoch 120/150 - Train MAE: 3.5237 | Test MAE: 3.6435 | LR: 0.000000
Epoch 130/150 - Train MAE: 3.5237 | Test MAE: 3.643

In [33]:
import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=3, rounding_step=0)

params = QuantitativeObservationTableParameters(
    tol_dist_init=6, 
    rec_method="direct", 
    r=3, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent_det()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_deterministic_wfa_opt()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=4, 
        random_max_len=15, 
        num_random=0,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1


--- Iteration 1 ---
[Exact L* Synthesis] Empirical Hankel DWFA successfully built with 1 states.
[OK] The Automaton satisfies the Twins Property. It is 100% determinizable!
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 4...
      -> Failure detail for 'WI': RNN=50.42, WFA=10.12

[!] Counterexample found (Exhaustive): 'WI'
[+] Processing Counterexample: 'WI' | S size: 1
[Counterexample] Added ALL suffixes to E from: 'WI'

--- Iteration 2 ---
[Exact L* Synthesis] Empirical Hankel DWFA successfully built with 5 states.
[OK] The Automaton satisfies the Twins Property. It is 100% determinizable!
  [EQ-Hybrid] Phase 1: Exhaustive search up to length 4...
      -> Failure detail for 'IR': RNN=17.05, WFA=9.58

[!] Counterexample found (Exhaustive): 'IR'
[+] Processing Counterexample: 'IR' | S size: 5
[Counterexample] Added ALL suffixes to E from: 'IR'

--- Iteration 3 ---
[Exact L* Synthesis] Empirical Hankel DWFA successfully built with 6 states.
[OK] The Automaton satisfies the Twin

In [35]:
#Manipulate weights

wfa.push_weights_to_positive()
wfa.normalize_global_bias()
wfa.push_final_weights_to_zero()


[+] Weight Pushing applied: Negative weights pushed towards the endpoints.
[+] Bias Normalized: Global weight shifted by -101.13.
[+] Backward Weight Pushing: Final weights absorbed to 0.0.


In [37]:
dot = draw_tropical_wfa(wfa)
dot.render('NoiseCase3', view=True,format='png')

'NoiseCase3.png'

In [3]:

ruta_guardado = "modelo_caso3_noise.pth"

torch.save(model.state_dict(), ruta_guardado)


In [7]:
modelo_recuperado = CostPredictorRNN(vocab_size=5, embed_dim=8, hidden_dim=32)

modelo_recuperado.load_state_dict(torch.load("modelo_caso3_noise.pth"))


modelo_recuperado.eval()

model = modelo_recuperado

C:\Users\rafam\AppData\Local\Temp\ipykernel_18188\3625650957.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modelo_recuperado.load_state_dict(torch.load("modelo_caso3.p